# Qwen3-VL Spatial Reasoning Circuit Benchmark

Thin driver notebook. All reusable logic lives in `src/` — this notebook just
calls into it, runs the pipeline top-to-bottom, and displays results/plots.

See `README.md` for the full pipeline description and repo structure.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))  # so `import src` works when run from notebooks/

import json
from IPython.display import display

from src import config as cfg
from src import benchmark as bm
from src import model_utils as mu
from src import patching
from src import analysis
from src import headlens
from src import plotting

cfg.set_seed()
print("DEVICE:", cfg.DEVICE)
print("MODEL:", cfg.MODEL_ID)
print("OUTPUT_DIR:", cfg.OUTPUT_DIR)

## 1. Load Qwen3-VL

In [ ]:
model, processor = mu.load_model_and_processor()
decoder_layers = mu.get_decoder_layers(model)
print("Decoder layers:", len(decoder_layers))

## 2. Benchmark design

We use a **4 × 3 grid with 12 unique objects**. Every cell contains one object, so "the object immediately above X" is unambiguous.

All five reasoning levels are evaluated on the **same visual scene**. Only the query's relational composition changes. For each query we search for a single object-swap corruption that flips the ground-truth answer (see `src/benchmark.py:find_flipping_corruption`).

## 3. Build the 10-scene benchmark

In [ ]:
import pandas as pd

BENCHMARK = bm.build_benchmark(n_samples=10, seed=cfg.SEED)

print("Scenes:", len(BENCHMARK))
print("Queries:", sum(len(s["queries"]) for s in BENCHMARK))

benchmark_df = pd.DataFrame(bm.benchmark_to_rows(BENCHMARK))
display(benchmark_df)

bm.save_benchmark(BENCHMARK, cfg.out_path("spatial_reasoning_benchmark_10.json"))
benchmark_df.to_csv(cfg.out_path("spatial_reasoning_benchmark_10.csv"), index=False)
print("Saved benchmark JSON + CSV to", cfg.OUTPUT_DIR)

## 4. Visual sanity check

In [ ]:
import matplotlib.pyplot as plt

scene = BENCHMARK[0]
clean_pos = bm.scene_to_positions(scene, corrupt=False)
corrupt_pos = bm.scene_to_positions(scene, corrupt=True, query_index=4)

fig, ax = plt.subplots(1, 2, figsize=(10, 4))
ax[0].imshow(bm.make_image(clean_pos)); ax[0].axis("off"); ax[0].set_title("Sample 1 — clean")
ax[1].imshow(bm.make_image(corrupt_pos)); ax[1].axis("off"); ax[1].set_title("Sample 1 — corrupted")
plt.tight_layout()
plt.show()

for q in scene["queries"]:
    print(f"L{q['level']}: {q['question']}")
    print("   GT:", q["clean_answer"], "-> corrupted:", q["corrupted_answer"])

## 5. Qwen input construction and token groups

In [ ]:
sample_q = BENCHMARK[0]["queries"][0]["question"]
sample_inputs = mu.build_inputs(processor, cfg.DEVICE, bm.make_image(clean_pos), sample_q)

ctx = patching.ModelContext.build(model, processor, device=cfg.DEVICE, sample_inputs=sample_inputs)
print(f"image_token_id={ctx.image_token_id}  NUM_HEADS={ctx.num_heads}  HEAD_DIM={ctx.head_dim}")

## 6. Baseline evaluation

Always run this before interpreting VAP. We need to know whether Qwen3-VL actually distinguishes clean and corrupted examples at each reasoning level.

In [ ]:
baseline_df, baseline_summary = patching.run_baseline_evaluation(ctx, BENCHMARK)
display(baseline_summary)
baseline_df.to_csv(cfg.out_path("baseline_by_instance.csv"), index=False)

## 7. Layer × token-group VAP

For every valid pair:

1. Cache corrupted hidden states.
2. Select a decoder layer and token group.
3. Replace the clean hidden state at that location with the corrupted state.
4. Measure whether the clean answer moves toward the corrupted answer.

This is the most expensive cell in the notebook — it runs one forward pass per (scene, query, layer, token group). Skip and reload `layer_vap_by_difficulty.csv` from a previous run if you just want to regenerate downstream plots/tables.

In [ ]:
RUN_LAYER_SWEEP = True  # set False to reload a previously saved layer_summary CSV instead

if RUN_LAYER_SWEEP:
    layer_df, layer_summary = patching.run_layer_vap_sweep(ctx, BENCHMARK, token_groups=cfg.TOKEN_GROUPS)
    layer_summary.to_csv(cfg.out_path("layer_vap_by_difficulty.csv"), index=False)
else:
    layer_summary = pd.read_csv(cfg.out_path("layer_vap_by_difficulty.csv"))

display(layer_summary.head(20))

## 8. Plot layer curves for L1-L5

In [ ]:
_ = plotting.plot_layer_curves(layer_summary, token_groups=cfg.TOKEN_GROUPS)

## 9. Candidate layers for head-level analysis

We use the `last_prompt_token` curve (among others) to select the top layers separately for L1–L5. This reduces the computational cost of the head sweep.

In [ ]:
candidate_layers_by_level = analysis.select_candidate_layers(layer_summary, top_layers_per_level=cfg.TOP_LAYERS_PER_LEVEL)
analysis.save_candidate_layers(candidate_layers_by_level, cfg.out_path("candidate_layers_by_level.json"))

## 10. Head-level VAP

Now the main experiment:

> **Are the same attention heads causally important for L1, L2, L3, L4 and L5?**

We patch one attention head at the answer-generating position and measure the reduction in the clean-vs-corrupted answer logit gap.

In [ ]:
head_df, head_results_df = patching.run_head_vap_sweep(ctx, BENCHMARK, candidate_layers_by_level)

head_df.to_csv(cfg.out_path("head_vap_raw_all_groups.csv"), index=False)
head_results_df.to_csv(cfg.out_path("head_vap_by_level_all_groups.csv"), index=False)

display(head_results_df.sort_values(["importance", "overwrite_rate"], ascending=[False, False]).head(15))

In [ ]:
# Sanity check
assert not head_results_df.empty, "head_results_df is empty — rerun the head sweep above."
print(f"head_results_df: {head_results_df.shape[0]} rows, levels={sorted(head_results_df['level'].unique())}")

## 11. Head-level causal importance — one figure for the paper

A single multi-panel figure: rows = reasoning level, columns = token group, shared colorbar and color scale.

In [ ]:
_ = plotting.plot_head_importance_grid(head_results_df, candidate_layers_by_level, num_heads=ctx.num_heads)

## 12. Does the same circuit appear at every difficulty?

Take the top-k heads for each difficulty and compute Jaccard overlap:

$$J(A,B)=\frac{|A\cap B|}{|A\cup B|}$$

High overlap means the same heads are reused; low overlap suggests circuit specialization. Point-estimate overlap alone is a weak claim for a paper, so we resample scenes with replacement (B=200) to get a 95% CI on each pairwise overlap.

In [ ]:
top_heads_by_level, overlap_mean, overlap_lo, overlap_hi = analysis.bootstrap_head_overlap(
    head_df, head_results_df, top_k=cfg.TOP_K_HEADS, n_boot=cfg.N_BOOT, seed=cfg.SEED
)

for lv, heads in top_heads_by_level.items():
    print(f"L{lv}: {sorted(heads)}")

overlap_df = analysis.overlap_matrix_to_df(overlap_mean)
overlap_df.to_csv(cfg.out_path("head_overlap_mean.csv"))

_ = plotting.plot_overlap_heatmap(overlap_mean, overlap_lo, overlap_hi, top_k_heads=cfg.TOP_K_HEADS, n_boot=cfg.N_BOOT)

## 13. Core vs. difficulty-specific heads — summary table (CSV + LaTeX)

In [ ]:
circuit_table = analysis.build_circuit_summary_table(top_heads_by_level)

analysis.save_circuit_summary(
    circuit_table,
    top_heads_by_level,
    csv_path=cfg.out_path("circuit_summary_table.csv"),
    tex_path=cfg.out_path("circuit_summary_table.tex"),
    json_path=cfg.out_path("circuit_summary.json"),
)

display(circuit_table)

## 14. Optional HeadLens

Use HeadLens only after the causal sweep. It provides directional evidence about which vocabulary a selected head's isolated residual contribution points toward.

Do **not** interpret the output as calibrated probabilities.

In [ ]:
headlens_results = headlens.run_headlens_for_scene(ctx, BENCHMARK[0], head_results_df, top_n_heads=5, top_k=8)

for level, rows in headlens_results.items():
    print(f"\n===== L{level} =====")
    for layer, head, importance, decoded in rows:
        print(f"L{layer} H{head} importance={importance:.4f}: {decoded}")

## 15. What to look for

The important result is not "Head X is important."

Compare:

1. **Layer stability:** do the same layers peak from L1 → L5?
2. **Head stability:** how large is the Jaccard overlap?
3. **Circuit expansion:** do harder queries recruit additional heads?
4. **Core circuit:** heads that appear in ≥4/5 levels' top-k list, per `circuit_summary_table.csv`.